In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

print(os.getenv("COMTRADE_API_KEY"))

In [ ]:
nat_gas_codes = {271111: 'Natural gas, liquefied',
                 271121: 'Natural gas, gaseous'}

year_list = list(range(2000, datetime.now().year)) # list of all years from 2000 to now

folder_name = 'nat_gas_2000-2025_Trade_data'


In [ ]:
dfr = pd.read_parquet(os.path.join(folder_name, 'nat_gas_trade_df'), engine='pyarrow')
print(dfr.shape)
print(dfr.shape)
dfr.sample(5).style

In [ ]:
dfr.groupby(['refYear', 'flow', 'type'])[['primaryValue', 'netWgt']].agg('sum')/10**9 
#total import and export in dollars in kg in billions for each form of gas

We see huge discrepancies in netWgt (traded gas measured in kg) in gaseous (and in liquified as appeared but smaller) form of gas starting from 2015 - netWgt needs filtering to use. Difference in export and import is expected - countries export goods with much lower prices then others import these goods. But difference in kg values is unreasonable.

In [ ]:
dfr[(dfr.flow == 'Import') & (dfr.refYear >= 2015)].groupby('refYear')['netWgt'].nlargest(5)/10**9

In [ ]:
bad_rows_g = dfr[ (dfr.cmdCode == 271121) & (dfr.flow == 'Import') & (dfr.refYear >= 2015)].groupby('refYear')['netWgt'].nlargest(7).index
bad_indexes_g = []
for i in range(len(bad_rows_g)):
    bad_indexes_g.append(bad_rows_g[i][1])      #getting indexes of these countries with crazy amounts of volume of trade in kg (7 just to see)

In [ ]:
needed_cols = ['period', 'reporter', 'flow', 'partner', 'cmdCode', 'primaryValue', 'qty', 'qty_type', 'netWgt', 'altQty', 'alt_qty_type']
dfr.loc[bad_indexes_g, needed_cols].sample(10)

In [ ]:
bad_rows_l = dfr[ (dfr.cmdCode == 271111) & (dfr.flow == 'Import') & (dfr.refYear >= 2015)].groupby('refYear')['netWgt'].nlargest(4).index
bad_indexes_l = []
for i in range(len(bad_rows_l)):
    bad_indexes_l.append(bad_rows_l[i][1])  #the same for liquified (it appeared there are also discrepancies - smaller than in gaseous so difficult to notice)

In [ ]:
dfr.loc[bad_indexes_l, needed_cols].sample(10)

In [ ]:
dfr_kg = dfr[ ~( (dfr.refYear >= 2015) & 
                 (dfr.reporter == 'Mexico') & (dfr.flow == 'Import') & (dfr.partner == 'United States of America') )]
dfr_kg.shape, dfr.shape

In [ ]:
dfr_kg.groupby(['refYear', 'flow', 'cmdCode'])[['primaryValue', 'netWgt']].agg('sum')/10**9

We see a common pattern: there is a mistake in registering value in kg of import of gaseous and liquified natural gas from USA to Mexico - so I created one more df without these trades and everything seems logical (it is likely that they write value in liters to value in kg since value in liters has the same value). However, trade value in dollars between countries corresponds to reality - I checked in public databases. So I will use for dollar analysis dfr (main df) and for analysis in kg I will use dfr_kg.

In [ ]:
def plot_trades(column:str, measure: str, df=dfr):
    for i in range(2):
        for fl in ['Export', 'Import']:
            plt.figure(figsize=(10,3))
            (df[(df.flow == fl) & (df.cmdCode == list(nat_gas_codes.keys())[i] )].groupby('refYear')[column].sum()/10**9).plot.bar()
            plt.title(nat_gas_codes[ list(nat_gas_codes.keys())[i] ] + '; ' + str(column) + ', Total ' + str(fl))
            plt.xlabel('Years')
            plt.ylabel('Total ' +str(fl) + ', billions of ' +str(measure))
            plt.grid(True, alpha=0.5)

In [ ]:
plot_trades(column='primaryValue', measure='dollars', df=dfr)

In [ ]:
plot_trades(column='netWgt', measure = 'kilograms', df=dfr_kg)

However, there a still problems with values of export of natural liquified gas in 2016, 2018, and 2020.

In [ ]:
dfr_kg[((dfr_kg.refYear == 2016) | (dfr_kg.refYear == 2018) | (dfr_kg.refYear == 2020)) 
       & (dfr_kg.flow == 'Export') & (dfr_kg.cmdCode == 271111)].groupby('refYear').netWgt.nlargest(6)

In [ ]:
bad_rows = dfr_kg[((dfr_kg.refYear == 2016) | (dfr_kg.refYear == 2018) | (dfr_kg.refYear == 2020)) 
       & (dfr_kg.flow == 'Export') & (dfr_kg.cmdCode == 271111)].groupby('refYear').netWgt.nlargest(6).index
bad_indexes = []
for i in range(len(bad_rows)):
    bad_indexes.append(bad_rows[i][1])      #the same to get indexes of these trades

In [ ]:
dfr_kg.loc[bad_indexes, needed_cols]

We again see discrepancies - mostly with Nigeria, but other countries also appear. 
There should not be too huge difference between netWgt (weight in kg) and primaryValue (value in dollars) as price of gas has always been in a range about 0.1$-3$ per kg. Trades with other prices are barely possible: either these are artificial trades or there is a mistake in reports. Let us filter out all trades of unrealistic prices to make a dataset robust for analysis.

I will filter them from the df already without import of Mexico from USA because they confused volume with liters with values in kilograms - it is another type of mistake. These trades are mostly filtered by price but some smaller not, so it is more robust to work without them.

In [ ]:
print(f'Original df shape: {dfr.shape}, df without Mexico-Usa trades: {dfr_kg.shape}')
dfr_kg = dfr_kg[dfr_kg.netWgt > 0] #to calculate price and not divide on 0 or have NaN.
print(f'DataFrame with prices shape: {dfr_kg.shape} \
      Lost rows: {dfr.shape[0] - dfr_kg.shape[0]} ({(dfr.shape[0] - dfr_kg.shape[0])*100/dfr.shape[0]:.4f}%)')

In [ ]:
dfr_kg['kg_price'] = dfr_kg.primaryValue / dfr_kg.netWgt

In [ ]:
dfr_kg['kg_price'].describe()

In [ ]:
dfr_kg['kg_price'].sample(10)

In [ ]:
low_lim = dfr_kg.kg_price.quantile(0.01)
high_lim = dfr_kg.kg_price.quantile(0.99)
low_lim, high_lim

In [ ]:
dfr_kg = dfr_kg[(dfr_kg.kg_price > low_lim) & (dfr_kg.kg_price < high_lim)]

In [ ]:
dfr_kg['kg_price'].describe()

In [ ]:
plot_trades(column='netWgt', measure = 'kilograms', df=dfr_kg)

In [ ]:
dfr_kg.to_parquet(os.path.join(folder_name, 'nat_gas_trade_df_kg'), engine='pyarrow', compression='snappy')

print(f'In total we lost {dfr.shape[0] - dfr_kg.shape[0]} ({(dfr.shape[0] - dfr_kg.shape[0])*100/dfr.shape[0]:.4f}%) rows to have df to work with weights.')

Now we have df to analyse trade in kilograms to see evolution of global trade without influence of prices.
These df is a bit truncated but still is robust to compare with df with dollar values. All huge outliers and mistakes are deleted.